In [1]:
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


SEGMENT = "region_code <= 20"          # двадцать регионов из восьмидесяти девяти
WINDOW = ("2026-01-01", "2026-07-01")  # окно фактов: полугодие, правый конец не включён

print("подключились как:", q("select current_user as login").iloc[0, 0])

подключились как: avgaliaminskaia


In [2]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by p.payment_id
""")

sub_members = q(f"""
    select m.subscription_id, m.client_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

In [3]:
members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

In [4]:
partners = q(f"""
    select *
    from prime.partners
""")

services = q(f"""
    select *
    from prime.services
""")

service_usage = q(f"""
    select * from prime.service_usage
    where {CLIENT_FILTER}
    """)

In [5]:
transactions = q(f"""
    select *
    from prime.transactions
    where {CLIENT_FILTER}
""")

cashback = q(f"""
    select *
    from prime.cashback
""")

support_tickets = q(f"""
    select * from prime.support_tickets
    where {CLIENT_FILTER}
""")

In [23]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import timedelta

# 1. Добавляем корневую папку проекта в sys.path, чтобы работали импорты из src
# os.path.abspath('') возвращает путь до папки notebooks, поднимаемся на уровень выше
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Импортируем функции из твоего файла data.py
# (замени названия функций, если они немного отличаются)
from src.prime.data import (
    get_daily_revenue_cached, 
    get_active_subscriptions_cached, 
    get_churn_cached, 
    get_new_clients_cached, 
    get_sessions_cached, 
    get_cbr_rate_cached
)

# 3. Импортируем проверки, если они тебе нужны (из checks.py)
try:
    import src.prime.checks as checks
    print("✅ Модуль checks.py успешно импортирован")
except ImportError:
    print("⚠️ Модуль checks.py не найден или содержит ошибки")

print("✅ Все функции из data.py готовы к работе!")

✅ Модуль checks.py успешно импортирован
✅ Все функции из data.py готовы к работе!


In [26]:
import pandas as pd
import numpy as np

class MetricAnalyzer:
    def __init__(self, start_date: str, end_date: str):
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date)
        self.metrics_df = pd.DataFrame()
        
    def fetch_data_for_period(self):
        dates = pd.date_range(self.start_date, self.end_date, freq='D')
        records = []
        
        print(f"Загрузка данных из БД/кэша с {self.start_date.date()} по {self.end_date.date()}...")
        
        # Просто обычный цикл без tqdm
        for date_obj in dates:
            date_str = date_obj.strftime('%Y-%m-%d')
            
            rev_df = get_daily_revenue_cached(date_str)
            revenue = rev_df['total_revenue'].sum() if not rev_df.empty else 0
            payments = rev_df['payments_count'].sum() if not rev_df.empty else 0
            
            subs_df = get_active_subscriptions_cached(date_str)
            active_subs = subs_df['active_subs'].sum() if not subs_df.empty else 0
            
            churn_df = get_churn_cached(date_str)
            churned = churn_df['churned_subs'].sum() if not churn_df.empty else 0
            
            new_clients_df = get_new_clients_cached(date_str)
            new_clients = new_clients_df['new_clients'].sum() if not new_clients_df.empty else 0
            
            sessions_df = get_sessions_cached(date_str)
            sessions = sessions_df['sessions_count'].sum() if not sessions_df.empty else 0
            
            records.append({
                'date': date_obj, 'revenue': revenue, 'payments': payments,
                'active_subs': active_subs, 'churned_subs': churned,
                'new_clients': new_clients, 'sessions': sessions
            })
            
        self.metrics_df = pd.DataFrame(records).set_index('date')
        print("\n✅ Данные загружены!")
        
    def calculate_metrics(self):
        if self.metrics_df.empty: return
            
        self.metrics_df['arppu'] = np.where(
            self.metrics_df['payments'] > 0, self.metrics_df['revenue'] / self.metrics_df['payments'], 0
        )
        
        self.metrics_df['active_subs_prev'] = self.metrics_df['active_subs'].shift(1)
        self.metrics_df['churn_rate'] = np.where(
            self.metrics_df['active_subs_prev'] > 0,
            self.metrics_df['churned_subs'] / self.metrics_df['active_subs_prev'], 0
        )
        
        self.metrics_df['sessions_per_sub'] = np.where(
            self.metrics_df['active_subs'] > 0, self.metrics_df['sessions'] / self.metrics_df['active_subs'], 0
        )
        
    def check_anomalies(self):
        if self.metrics_df.empty: return
            
        self.metrics_df['revenue_ma7'] = self.metrics_df['revenue'].rolling(window=7, min_periods=1).mean()
        self.metrics_df['sessions_ma7'] = self.metrics_df['sessions'].rolling(window=7, min_periods=1).mean()
        
        self.metrics_df['alert_revenue_drop'] = self.metrics_df['revenue'] < (self.metrics_df['revenue_ma7'] * 0.8)
        self.metrics_df['alert_sessions_drop'] = self.metrics_df['sessions'] < (self.metrics_df['sessions_ma7'] * 0.75)
        self.metrics_df['alert_high_churn'] = self.metrics_df['churn_rate'] > 0.02
        
        alerts = self.metrics_df[
            self.metrics_df['alert_revenue_drop'] | 
            self.metrics_df['alert_sessions_drop'] | 
            self.metrics_df['alert_high_churn']
        ]
        
        if not alerts.empty:
            print(f"\n⚠️ НАЙДЕНО {len(alerts)} ДНЕЙ С АНОМАЛИЯМИ:")
            for date, row in alerts.iterrows():
                print(f"Дата: {date.strftime('%Y-%m-%d')} | Выручка: {row['revenue']} | Отток: {row['churn_rate']*100:.2f}%")
        else:
            print("\n✅ Аномалий не обнаружено.")

    def run(self):
        self.fetch_data_for_period()
        self.calculate_metrics()
        self.check_anomalies()
        return self.metrics_df

In [27]:
calc = MetricsCalculator(clients, subs, pay, spend, registrations)

print(" Дневная выручка")
print(calc.get_daily_revenue().tail())

 Дневная выручка
    report_date   revenue          ma_7d   drop_pct
595  2026-08-26  191710.0  198890.000000  -3.610036
596  2026-08-27  184330.0  191020.142857  -3.502323
597  2026-08-28  199339.0  190515.142857   4.631578
598  2026-08-29  182073.0  188853.857143  -3.590531
599  2026-08-30   84993.0  172831.285714 -50.823140


In [ ]:
print("\n LTV / CAC по каналам")
print(calc.get_ltv_cac())


--- LTV / CAC по каналам ---
           channel          cac          ltv  ltv_cac_ratio
0              crm   213.471404  3803.724667          17.82
1          organic   137.113335  4049.794660          29.54
2  partner_telecom   676.141057  3551.837156           5.25
3      performance  1528.485206  3327.956938           2.18
4         referral   349.528589  3897.745047          11.15
5           social   809.858008  3029.518510           3.74
